In [1]:
import sys
sys.path.insert(0, "/home/syedkazim/sciebo - Kazim, Syed Muhammad (u491036@uni-siegen.de)@uni-siegen.sciebo.de/Lab/Projects/2024_Phase_Camera_FM_Design/coded_wfs_sim")

from coded_wfs_sim import geometry
from coded_wfs_sim import propagator
from coded_wfs_sim import visualization
from coded_wfs_sim import utils

import numpy as np
from tifffile import tifffile
from matplotlib import pyplot as plt
import cv2

from scipy.signal import correlate2d
from scipy.interpolate import RegularGridInterpolator

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


* **Multiple Cubes on a Plane**
    1. Specify number of elements and tteratively sample positions
    2. Discretize the grid and probabilistically sample element at each position 

In [2]:
# Grid and propagation parameters setup
wl = 640e-9
spatial_resolution = [500e-9, 500e-9, 50e-9] # dx, dy, dz
grid_shape = [500, 500, 500] # x=0->, y=0->, z=0->
n_background = 1. # immersion medium RI
spatial_support = [spatial_resolution[i]*grid_shape[i] for i in range(3)]

In [ ]:
# Method 1

geom = geometry.Geometry(grid_shape, spatial_resolution, n_background)

plane_pnt = [0, 0, 12e-6]
plane_normal = [0, 0, 1]

num = 125 # create num elemets
for i in range(num):
    print(f"Shapes: {i+1}/{num}", end="\r")
    pos_x, pos_y = np.random.uniform(6, 244, size=2)
    pos_x, pos_y = pos_x*1e-6, pos_y*1e-6
    geom.add_obj_on_plane('cube', (pos_x, pos_y), length=10e-6, RI=1.4609, 
                              plane=[plane_pnt, plane_normal], bias=2.75e-6)

geom.add_plane(point=plane_pnt, normal=plane_normal, RI=1.49, thickness=15e-6)

# Retrieve 3d RI distribution
RI_distribution = geom.get_grid()
print('Geometry: Done')

# visualization
visualization.visualize_grid_vol(RI_distribution[::2, ::2, ::2], n_background=n_background, factor=1)


del geom

Coordiante system with size: 
 
              X = [0, 2.50e-04], Res_X = 5e-07
              Y = [0, 2.50e-04], Res_Y = 5e-07
              Z = [0, 2.50e-05], Res_Z = 5e-08
              Immersion RI: 1.0
      
Geometry: Done5


In [3]:
# Method 2

geom = geometry.Geometry(grid_shape, spatial_resolution, n_background)
pos = geom.unifrom_plane_sampling_positions(10e-6, prob=0.6)

plane_pnt = [0, 0, 12e-6]
plane_normal = [0, 0, 1]

for i in range(pos.shape[0]):
    print(f"Shapes: {i+1}/{pos.shape[0]}", end="\r")
    geom.add_obj_on_plane('cube', (pos[i, 0], pos[i, 1]), length=10e-6, RI=1.4609, 
                              plane=[plane_pnt, plane_normal], bias=2.75e-6)

geom.add_plane(point=plane_pnt, normal=plane_normal, RI=1.49, thickness=15e-6)

# Retrieve 3d RI distribution
RI_distribution = geom.get_grid()
print('Geometry: Done')

# visualization
visualization.visualize_grid_vol(RI_distribution[::2, ::2, ::2], n_background=n_background, factor=1)


del geom

Coordiante system with size: 
 
              X = [0, 2.50e-04], Res_X = 5e-07
              Y = [0, 2.50e-04], Res_Y = 5e-07
              Z = [0, 2.50e-05], Res_Z = 5e-08
              Immersion RI: 1.0
      
Geometry: Done1
